In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import datetime as dt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

import warnings
warnings.simplefilter(action="ignore")

pd.set_option('display.max_columns',1000)
pd.set_option('display.width', 500)
pd.set_option('display.float_format',lambda x : '%.2f' % x)

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
df_ = pd.read_csv("data/dataset.csv", compression="gzip")
df = df_.copy()
df.head()

,RecipeId,Name,CookTime,PrepTime,TotalTime,RecipeIngredientParts,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions
0,38,Low-Fat Berry Blue Frozen Dessert,1440,45,1485,"c(""blueberries"", ""granulated sugar"", ""vanilla ...",170.90,2.50,1.30,8.00,29.80,37.10,3.60,30.20,3.20,"c(""Toss 2 cups berries with sugar."", ""Let stan..."
1,41,Carina's Tofu-Vegetable Kebabs,20,1440,1460,"c(""extra firm tofu"", ""eggplant"", ""zucchini"", ""...",536.10,24.00,3.80,0.00,1558.60,64.20,17.30,32.10,29.30,"c(""Drain the tofu, carefully squeezing out exc..."
2,42,Cabbage Soup,30,20,50,"c(""plain tomato juice"", ""cabbage"", ""onion"", ""c...",103.60,0.40,0.10,0.00,959.30,25.10,4.80,17.70,4.30,"c(""Mix everything together and bring to a boil..."
3,45,Buttermilk Pie With Gingersnap Crumb Crust,50,30,80,"c(""sugar"", ""margarine"", ""egg"", ""flour"", ""salt""...",228.00,7.10,1.70,24.50,281.80,37.50,0.50,24.70,4.20,"c(""Preheat oven to 350°F."", ""Make pie crust, u..."
4,46,A Jad - Cucumber Pickle,0,25,25,"c(""rice vinegar"", ""haeo"")",4.30,0.00,0.00,0.00,0.70,1.10,0.20,0.20,0.10,"c(""Slice the cucumber in four lengthwise, then..."


In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
def grab_col_names(dataframe, cat_th=10, car_th=20):

    cat_cols = [col for col in dataframe.columns if dataframe[col].dtypes == "O"]
    num_but_cat = [col for col in dataframe.columns if dataframe[col].nunique() < cat_th and
                   dataframe[col].dtypes != "O"]
    cat_but_car = [col for col in dataframe.columns if dataframe[col].nunique() > car_th and
                   dataframe[col].dtypes == "O"]
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    # num_cols
    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes != "O"]
    num_cols = [col for col in num_cols if col not in num_but_cat]

    print(f"Observations: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f'cat_cols: {len(cat_cols)}')
    print(f'num_cols: {len(num_cols)}')
    print(f'cat_but_car: {len(cat_but_car)}')
    print(f'num_but_cat: {len(num_but_cat)}')
    return cat_cols, num_cols, cat_but_car

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
cat_cols, num_cols, num_but_cat = grab_col_names(df)

Observations: 375703
Variables: 16
cat_cols: 0
num_cols: 13
cat_but_car: 3
num_but_cat: 0


In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
def outlier_thresholds(dataframe, col_name, q1=0.01, q3=0.99):
    quartile1= dataframe[col_name].quantile(q1)
    quartile3= dataframe[col_name].quantile(q3)
    interquantile_range = quartile3 -quartile1
    up_limit= quartile3 +1.5 * interquantile_range
    low_limit= quartile1 -1.5 * interquantile_range
    return low_limit, up_limit

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit

for col in num_cols:
    replace_with_thresholds(df, col)

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
def check_outlier(dataframe, col_name):
    low_limit, up_limit = outlier_thresholds(dataframe, col_name)
    if dataframe[(dataframe[col_name] > up_limit) | (dataframe[col_name] < low_limit)].any(axis=None):
        return True
    else:
        return False

check_outlier(df,num_cols)

False

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
df= df.iloc[:,1:]

In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from yellowbrick.cluster import KElbowVisualizer
from scipy.cluster.hierarchy import linkage
from scipy.cluster.hierarchy import dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import AgglomerativeClustering

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
cat_cols, num_cols, num_but_cat = grab_col_names(df)

Observations: 375703
Variables: 15
cat_cols: 0
num_cols: 12
cat_but_car: 3
num_but_cat: 0


In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
df2=df.copy()

In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
sc = MinMaxScaler((0, 1))
df2[num_cols] = sc.fit_transform(df2[num_cols])

In [13]:
# --- [CELL 12]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 13}
kmeans = KMeans(n_clusters=30, n_init="auto").fit(df2[["TotalTime","Calories","SugarContent"]])

In [14]:
# --- [CELL 13]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 14}
clusters_kmeans = kmeans.labels_
clusters_kmeans

array([28, 28, 19, ..., 25, 25, 12], dtype=int32)

In [15]:
# --- [CELL 14]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 15}
df["kmeans_cluster"] = clusters_kmeans
df["kmeans_cluster"]= df["kmeans_cluster"] + 1
df.head()

,Name,CookTime,PrepTime,TotalTime,RecipeIngredientParts,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions,kmeans_cluster
0,Low-Fat Berry Blue Frozen Dessert,1200,45,1485.00,"c(""blueberries"", ""granulated sugar"", ""vanilla ...",170.90,2.50,1.30,8.00,29.80,37.10,3.60,30.20,3.20,"c(""Toss 2 cups berries with sugar."", ""Let stan...",29
1,Carina's Tofu-Vegetable Kebabs,20,600,1460.00,"c(""extra firm tofu"", ""eggplant"", ""zucchini"", ""...",536.10,24.00,3.80,0.00,1558.60,64.20,17.30,32.10,29.30,"c(""Drain the tofu, carefully squeezing out exc...",29
2,Cabbage Soup,30,20,50.00,"c(""plain tomato juice"", ""cabbage"", ""onion"", ""c...",103.60,0.40,0.10,0.00,959.30,25.10,4.80,17.70,4.30,"c(""Mix everything together and bring to a boil...",20
3,Buttermilk Pie With Gingersnap Crumb Crust,50,30,80.00,"c(""sugar"", ""margarine"", ""egg"", ""flour"", ""salt""...",228.00,7.10,1.70,24.50,281.80,37.50,0.50,24.70,4.20,"c(""Preheat oven to 350°F."", ""Make pie crust, u...",26
4,A Jad - Cucumber Pickle,0,25,25.00,"c(""rice vinegar"", ""haeo"")",4.30,0.00,0.00,0.00,0.70,1.10,0.20,0.20,0.10,"c(""Slice the cucumber in four lengthwise, then...",13


In [16]:
# --- [CELL 15]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 16}
# === BEFORE (original) ===
# df.groupby('kmeans_cluster').agg({1: ['count','mean', 'median', 'sum'],
#                                     2: ['count','mean', 'median', 'sum'],
#                                     3: ['count','mean', 'median', 'sum'],
#                                     4: ['count','mean','median', 'sum']})

# === AFTER (edited) ===
df.groupby('kmeans_cluster').agg({"TotalTime": ['count','mean', 'median', 'sum'],
                                    "Calories": ['count','mean', 'median', 'sum'],
                                    "SugarContent": ['count','mean','median', 'sum']})

TotalTime                            Calories                           SugarContent                       
                   count    mean  median        sum    count   mean median         sum        count  mean median       sum
kmeans_cluster                                                                                                            
1                   2539  451.39  440.00 1146087.00     2539 310.91 295.80   789388.00         2539 12.91  12.40  32782.90
2                  23067   39.33   30.00  907299.00    23067 127.36 124.70  2937885.50        23067  6.23   6.20 143635.90
3                   1250  476.63  480.00  595790.00     1250 328.36 310.90   410452.50         1250 26.26  25.70  32823.20
4                  12703   48.90   40.00  621165.00    12703 230.34 231.60  2925958.90        12703 20.43  20.40 259576.80
5                  24362   47.85   40.00 1165629.00    24362 422.55 409.20 10294206.10        24362  1.69   1.70  41172.10
6                   9144   60.02   45.00  548867.00     9144 579.60 571.50  5299905.40         9144  9.17   9.00  83823.30
7                   3592  403.33  380.00 1448754.00     3592 458.72 438.70  1647704.40         3592  4.70   4.70  16885.80
8                    290 1828.56 1834.50  530281.00      290 333.02 252.80    96576.10          290 23.87  22.95   6921.50
9                  39975   39.27   30.00 1569947.00    39975 233.35 229.30  9328101.90        39975  1.31   1.30  52236.70
10                 17837   42.19   32.00  752623.00    17837 175.67 171.90  3133498.20        17837 12.87  12.80 229554.90
11                  7577   52.99   40.00  401540.00     7577 757.40 723.40  5738832.30         7577  3.68   3.60  27849.70
12                  2270 1599.25 1490.00 3630295.00     2270 230.14 181.90   522407.70         2270  3.91   2.80   8867.80
13                 38797   29.78   20.00 1155461.00    38797  69.70  68.90  2704168.00        38797  0.67   0.50  25875.10
14                  9536   55.73   45.00  531475.00     9536 436.40 422.40  4161534.90         9536 13.85  13.70 132039.70
15                  7967   53.84   45.00  428924.00     7967 302.05 305.10  2406429.30         7967 33.02  33.00 263074.30
16                 18292   42.28   30.00  773385.00    18292 136.09 132.70  2489308.30        18292  9.42   9.40 172280.70
17                  6411  254.46  250.00 1631331.00     6411 169.69 168.80  1087908.40         6411  2.80   2.60  17933.90
18                  2562   71.82   45.00  184007.00     2562 916.23 865.95  2347387.10         2562 13.11  13.00  33588.30
19                  5061   61.72   45.00  312341.00     5061 541.96 522.50  2742865.00         5061 20.51  20.30 103780.30
20                 14901   44.23   35.00  659074.00    14901 200.71 197.90  2990718.80        14901 16.69  16.80 248768.40
21                  9271   51.89   45.00  481090.00     9271 323.94 322.70  3003239.50         9271 28.48  28.50 264013.90
22                 17675   50.68   40.00  895807.00    17675 490.78 484.10  8674540.90        17675  4.89   4.80  86474.00
23                 18733   55.98   45.00 1048636.00    18733 334.54 332.40  6266942.50        18733  8.37   8.20 156884.20
24                  1827   70.15   45.00  128159.00     1827 775.02 713.70  1415954.00         1827 30.36  30.00  55474.70
25                  6850   64.02   50.00  438541.00     6850 380.25 380.70  2604699.00         6850 37.40  37.50 256182.20
26                 11100   48.42   40.00  537491.00    11100 248.92 253.00  2763005.50        11100 24.66  24.70 273693.70
27                 28718   35.24   30.00 1012069.00    28718 111.38 113.90  3198550.60        28718  3.29   3.30  94612.10
28                 29880   46.03   40.00 1375471.00    29880 293.01 292.60  8755269.60        29880  4.33   4.20 129256.10
29                   566 1433.18 1455.00  811178.00      566 319.43 271.10   180798.60          566 23.36  22.00  13223.80
30                  2950  585.96  515.50 1728593.00     2950 208.36 209.50   61

In [17]:
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
assert 'kmeans_cluster' in numeric_columns, 'Expected kmeans_cluster to be a numeric grouping column.'
numeric_columns.remove('kmeans_cluster')
assert len(numeric_columns) >= 5, 'Expected at least five numeric feature columns for this aggregation test.'

agg_result = df.groupby('kmeans_cluster').agg({
    numeric_columns[1]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[2]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[3]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[4]: ['count', 'mean', 'median', 'sum'],
})
assert not agg_result.empty, 'Grouped aggregation should produce a non-empty result.'
assert set(['count', 'mean', 'median', 'sum']).issubset(set(agg_result.columns.get_level_values(1)))

try:
    df.groupby('kmeans_cluster').agg({
        1: ['count', 'mean', 'median', 'sum'],
        2: ['count', 'mean', 'median', 'sum'],
        3: ['count', 'mean', 'median', 'sum'],
        4: ['count', 'mean', 'median', 'sum'],
    })
except KeyError:
    pass
else:
    raise AssertionError('Bug regression: integer-labeled aggregation keys unexpectedly succeeded.')